In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [5]:
file_path="pharmacy_data.xlsx"

In [6]:
fact=pd.read_excel(file_path,sheet_name="FactSales")
date_dim=pd.read_excel(file_path,sheet_name="DimDate")
pharmacy_dim=pd.read_excel(file_path,sheet_name="DimPharmacy")
product_dim=pd.read_excel(file_path,sheet_name="DimProduct")

In [ ]:
print("FactSales shape:",fact.shape)
print("DimDate shape:",date_dim.shape)
print("DimPharmacy shape:",pharmacy_dim.shape)
print("DimProduct shape:",product_dim.shape)

In [ ]:
fact.head()

In [ ]:
fact.isnull().sum()

In [ ]:
fact.duplicated().sum()

In [ ]:
fact.info()

In [ ]:
date_dim.head()

In [ ]:
date_dim.duplicated().sum()

In [ ]:
date_dim.info()

In [ ]:
pharmacy_dim.head()

In [ ]:
pharmacy_dim.duplicated().sum()

In [ ]:
pharmacy_dim.info()

In [ ]:
product_dim.duplicated().sum()

In [ ]:
product_dim.head()

In [ ]:
product_dim.duplicated().sum()

In [ ]:
product_dim.info()

- No Missing values detected across all tables.
- No duplicate records found.
- All required columns were already in appropriate data formats.

In [ ]:
fact[~fact["ProductID"].isin(product_dim["ProductID"])].shape[0]

In [ ]:
fact[~fact["PharmacyID"].isin(pharmacy_dim["PharmacyID"])].shape[0]

In [ ]:
fact[~fact["DateKey"].isin(date_dim["DateKey"])].shape[0]

In [ ]:
fact[fact["RevenueEUR"]<0].shape[0]

In [ ]:
fact[fact["MarginEUR"]<0].shape[0]

In [ ]:
fact[fact["MarginEUR"]>fact["RevenueEUR"]].shape[0]

In [ ]:
fact["CalculatedMargin"]=fact["RevenueEUR"]-fact["CostEUR"]
(fact["CalculatedMargin"]-fact["MarginEUR"]).abs().round().max()

In [ ]:
fact=fact.drop(columns=["CalculatedMargin"])

- All ProductID values in the fact table exist in the product dimension table.
- All PharmacyID values in the fact table exist in the pharmacy dimension table.
- All DateKey values in the fact table exist in the date dimension table.
- No orphan records detected.
- No negative revenue values.
- No negative margin values.
- No logical inconsistencies where margin exceeds revenue.
- MarginEUR is consistent with RevenueEUR − CostEUR.

In [ ]:
sales=fact.merge(pharmacy_dim,on="PharmacyID",how="left")
sales=sales.merge(product_dim,on="ProductID",how="left")
sales=sales.merge(date_dim,on="DateKey",how="left")
sales.head()

In [ ]:
print("Total Revenue:",sales["RevenueEUR"].sum())
print("Total Margin:",sales["MarginEUR"].sum())
print("Total Units Sold:",sales["UnitsSold"].sum())

In [ ]:
margin_percent=sales["MarginEUR"].sum()/sales["RevenueEUR"].sum()
print("Overall Margin %:",round(margin_percent*100,2))

In [ ]:
country_margin=sales.groupby("Country")[["RevenueEUR","MarginEUR"]].sum()
country_margin["Margin %"]=(country_margin["MarginEUR"]/country_margin["RevenueEUR"])*100
country_margin.sort_values("Margin %",ascending=False)

- The dataset generated 8.63M EUR in total revenue and 2,42M EUR in total margin, with 445,793 units sold.
- The overall margin percentage is ~28%, indicating relatively stable profitability across sales.
- Margin percentage is highly consistent across all countries (~28%), indicating a standardized pricing and cost structure.
- Revenue and profit differences are primarily driven by sales volume rather than profitability variation. Germany, France, and Italy are the largest contributors to total margin due to higher revenue levels.

In [ ]:
plt.hist(sales["RevenueEUR"],bins=30)
plt.title("Revenue Distribution")
plt.show()

In [ ]:
sales[["UnitsSold","ProductName","PharmacyName","RevenueEUR"]].sort_values("RevenueEUR",ascending=False).head(10)

In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(sales["RevenueEUR"])
plt.title("Revenue Boxplot")
plt.show()

In [ ]:
sales["RevenueEUR"].describe()

In [ ]:
plt.hist(sales["UnitsSold"],bins=30)
plt.title("Units Sold Distribution")
plt.show()

In [ ]:
plt.boxplot(sales["UnitsSold"])
plt.title("Units Sold Boxplot")
plt.show()

In [ ]:
plt.hist(sales["MarginEUR"],bins=30)
plt.title("Margin distribution")
plt.show()

In [ ]:
plt.boxplot(sales["MarginEUR"])
plt.title("Margin Boxplot")
plt.show()

- The distributions for Revenue, Units sold, and Margin are right-skewed.
- This indicates that most transactions are relatively small, while a few transactions generate very high sales values.
- The boxplots also reveal the presence of high-value outliers, which contribute significantly to total revenue and profit.

In [ ]:
date_dim.columns

In [ ]:
monthly_revenue=sales.groupby(["Year","MonthName"])["RevenueEUR"].sum().reset_index()

In [ ]:
monthly_revenue["YearMonth"]=pd.to_datetime(monthly_revenue["Year"].astype(str)+"-"+monthly_revenue["MonthName"].astype(str))

In [ ]:
monthly_revenue=monthly_revenue.sort_values(["YearMonth"],ascending=False)

In [ ]:
monthly_revenue.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(monthly_revenue["YearMonth"],monthly_revenue["RevenueEUR"])
plt.xticks(rotation=45)
plt.title("Monthly Revenue Trend")
plt.xlabel("Year-Month")
plt.ylabel("Revenue (EUR)")
plt.tight_layout()
plt.show()

In [ ]:
monthly_margin=sales.groupby("YearMonth")["MarginEUR"].sum()
plt.figure(figsize=(8,4))
monthly_margin.plot()
plt.title("Monthly Margin Trend")
plt.ylabel("Total Margin")
plt.show()

In [ ]:
monthly_analysis=sales.groupby("YearMonth").agg({"RevenueEUR":"sum","MarginEUR":"sum"})
monthly_analysis["MarginPercent"]=(monthly_analysis["MarginEUR"]/monthly_analysis["RevenueEUR"])*100
monthly_analysis["MarginPercent"].plot()
plt.title("Monthly Margin % Trend")
plt.ylabel("Margin %")
plt.show()

- Average monthly revenue varies across months, indicating that sales performance is not consistent throughout the year.
- Monthly margin shows noticeable fluctuations over time, with several peak months indicating higher profitability. The pattern of margin generally follows the revenue trend, suggesting that months with higher sales also generate higher profit margins.
- However, without multiple years of repeated patterns, these variations cannot be confidently attributed to seasonality. 
- The monthly margin percentage remains relatively stable across the observed period, fluctuating slightly around ~28%. This suggests that profitability per sale remains consistent over time, even when revenue changes.

In [ ]:
product_revenue=sales.groupby("ProductName")["RevenueEUR"].sum().sort_values(ascending=False)
product_revenue.head(10).plot(kind="bar")
plt.title("Top 10 Products by Revenue")
plt.ylabel("Total Revenue")
plt.show()

In [ ]:
product_margin=sales.groupby("ProductName")["MarginEUR"].sum().sort_values(ascending=False)
product_margin.head(10).plot(kind="bar")
plt.title("Top 10 Products by Margin")
plt.ylabel("Margin")
plt.show()

In [ ]:
product_analysis=sales.groupby("ProductName").agg({"RevenueEUR":"sum","MarginEUR":"sum"}).sort_values(by="RevenueEUR",ascending=False)
top_products=product_analysis.head(10)
plt.figure(figsize=(10,6))
top_products[["RevenueEUR","MarginEUR"]].plot(kind="barh")
plt.title("Top 10 Products:Revenue vs Margin")
plt.xlabel("Amount (EUR)")
plt.ylabel("Product")
plt.show()

In [ ]:
product_analysis["MarginPercent"]=(product_analysis["MarginEUR"]/product_analysis["RevenueEUR"])*100
product_analysis.sort_values("MarginPercent",ascending=False).head()

In [ ]:
top_products=product_analysis.sort_values("RevenueEUR",ascending=False).head(10)
plt.figure(figsize=(10,6))
plt.barh(top_products.index,top_products["MarginPercent"])
plt.xlabel("Margin")
plt.title("Margin % of Top 10 Revenue-Generating Products")
plt.tight_layout()
plt.show()

In [ ]:
product_analysis["MarginContribution"]=(product_analysis["MarginEUR"]/product_analysis["MarginEUR"].sum())*100
product_analysis.sort_values("MarginPercent",ascending=False).head()

In [ ]:
product_analysis.sort_values("MarginEUR",ascending=False).head()

- AntiBioX ACE Inhibitor 400 mg is the top-performing product in terms of revenue, generating the highest sales among all products. It also contributes the largest share of total margin, making it a major profit driver.
- NeuroMed Antidepressant 100 mg and DermRX Insulin Pen 200 mg also generate high revenue and margin contribution, indicating strong demand and significant impact on overall business performance.
- Some products such as NatureFit Omega-3 Plus and BioBalance Multivitamin Active show higher margin percentages (~37-38%), suggesting these products are more profitable relative to their sales.
- In contrast, products like DermRX Insulin Pen 200 mg and NeuroMed Antidepressant 100 mg have lower margin percentages (~21-22%) despite strong revenue, indicating higher costs or lower pricing margins.
- Overall, the top revenue products also contribute significantly to total profit, but margin percentage varies acoss products, showing differences in profitability efficiency.

In [ ]:
pharmacy_dim["Country"].value_counts().plot(kind="bar")
plt.title("Number of Pharmacies by Country")
plt.show()

In [ ]:
pharmacy_dim["PharmacyType"].value_counts().plot(kind="pie",autopct="%1.1f%%")
plt.title("Distribution of Pharmacies by Type")
plt.show()

In [ ]:
pharmacy_dim["StoreSizeBand"].value_counts().plot(kind="bar")
plt.title("Store Size Band Distribution")
plt.show()

In [ ]:
pharmacy_dim["OpenDate"]=pd.to_datetime(pharmacy_dim["OpenDate"])
pharmacy_dim["StoreAge"]=2025-pharmacy_dim["OpenDate"].dt.year
pharmacy_dim["StoreAge"].hist()
plt.title("Distribution of Store Age")
plt.show()

In [ ]:
pharmacy_dim["OpenYear"]=pharmacy_dim["OpenDate"].dt.year
pharmacy_dim["OpenYear"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of stores Openend per Year")
plt.show()

- Germany has the highest number of pharmacies, followed by France and italy, indicating stronger market presence in these countries.
- Netherlands and Australia have the lowest number of pharmacies, suggesting relatively small market coverage in the dataset.
- Most pharmacies are located in Urban(41.7%), and Suburhan(39.2%), indicating higher pharmacy presence in more populated regions.
- Rural areas(19.2%) have significantly fewer pharmacies, suggesting lower healthcare retail coverage compared to urban and suburhan regions.
- Medium-sized(M) pharmacies are the most common in the dataset, followed by small(S) pharmacies, while large(L) pharmacies are the least common.
- This indicates that the pharmacy network is mainly composed of medium and small stores, with fewer large outlets.
- The store age distribution shows that most pharmacies have been operating for around 3-5 years and 13-15 years, indicating a mix of relatively new and well-established stores.
- Fewer pharmacies fall in the mid-age ranges, suggesting uneven expansion over time.
- The number of stores opened fluctuates across years, with noticeable peaks around 2010,2021, and 2022, indicating periods of higher expansion.
- Store openings declined slightly in recent years(2023-2024), suggesting a slowdown in expnsion after the peak period.

In [ ]:
print(date_dim.shape)
print(date_dim["Date"].min())
print(date_dim["Date"].max())
print(date_dim["Year"].nunique())
print(date_dim["YearMonth"].nunique())

In [ ]:
date_dim.groupby("Year").size()

In [ ]:
date_dim.groupby(["Year","Quarter"]).size()

In [ ]:
date_dim.groupby("YearMonth").size()

In [ ]:
sales.groupby("Year")[["RevenueEUR", "MarginEUR", "UnitsSold"]].sum()

In [ ]:
yearly = sales.groupby("Year")["RevenueEUR"].sum()

growth = (yearly.loc[2025] - yearly.loc[2024]) / yearly.loc[2024] * 100
growth

In [ ]:
yearly_margin=sales.groupby("Year")["MarginEUR"].sum()
margin_growth=(yearly_margin.loc[2025]-yearly_margin.loc[2024])/yearly_margin.loc[2024]*100
margin_growth

In [ ]:
sales.groupby("Year").apply(
    lambda x: x["MarginEUR"].sum() / x["RevenueEUR"].sum() * 100
)

- From 2024 to 2025, the company shows positive growth across revenue, margin, and units sold.
- From 2024 to 2025, the business experienced positive growth, with revenue increasing by ~4.43% and margin increasing by ~4.93%, indicating steady improvement in both sales and profitability.
- Profit margin improved slightly in 2025, rising from 27.97% to 28.11%, suggesting better profit efficiency compared to the previous year.

In [ ]:
sales.columns

In [ ]:
if set(['StoreSizeBand','Region','RevenueEUR']).issubset(sales.columns):
    pivot = pd.pivot_table(sales, values='RevenueEUR', index='StoreSizeBand', columns='Category', aggfunc='sum')
    plt.figure(figsize=(8,5))
    sns.heatmap(pivot, annot=True, cmap='YlGnBu', fmt='.0f')
    plt.title('Revenue by Store size and Category')
    plt.show()

In [ ]:
if set(['StoreSizeBand','Region','RevenueEUR']).issubset(sales.columns):
    pivot = pd.pivot_table(sales, values='RevenueEUR', index='StoreSizeBand', columns='Category', aggfunc='mean')
    plt.figure(figsize=(8,5))
    sns.heatmap(pivot, annot=True, cmap='YlGnBu', fmt='.0f')
    plt.title('Average Revenue by Store size and Category')
    plt.show()

- Across small,medium, and large stores, the Prescription category leads in both total revenue and average per store, indicating that it is the most dominant and consistently performing revenue stream regardless of store size.

In [ ]:
if set(['Category','RevenueEUR','MarginEUR']).issubset(sales.columns):
    fig = px.scatter(sales, x='RevenueEUR', y='MarginEUR', color='Category', size='UnitsSold',
                     hover_data=['Brand','Country','Region'], title='Interactive Revenue vs Margin by Category')
    fig.show()

In [ ]:
sales['MarginPct']=sales['MarginEUR']/sales['RevenueEUR']
if set(['Category','RevenueEUR','MarginEUR']).issubset(sales.columns):
    fig = px.scatter(sales, x='RevenueEUR', y='MarginPct', color='Category', size='UnitsSold',
                     hover_data=['Brand','Country','Region'], title='Interactive Revenue vs Margin% by Category')
    fig.show()

In [ ]:
sales.groupby('Category')['UnitsSold'].sum().sort_values(ascending=False)

- A strong relationship exists between revenue and margin across all categories, indicating that higher sales directly contribute to higher total profit.
- Prescription products generate the highest revenue and total margin,making them the primary driver of overall business performance, largely due to high sales volume.
- Wellness products show the highest margin percentage, indicating superior profitability per unit of revenue despite lower overall sales volume.
- OTC products contribute the highest sales volume (units sold), reflecting strong and frequent customer demand, though with moderate profitability.

# **Project Overview**
This project analyzes pharmacy sales data to identify key drivers of revenue,product performance, and business growth between 2024-2025
## *1. Sales Performance Distribution*
Revenue, units sold, and margin distributions are right-skewed, indicating that most transactions generate moderate sales while a smaller nmber of transactions contribute significantly higher values.
## *2. Monthly Sales Trends*
Monthly revenue and margin show variation across months, suggesting that sales performance fluctuates over time rather than remaining constant.
## *3.Product Performance*
A small group of products generate a large share of total revenue and profit, highlighting the importance of key products in driving overall business performance.
## *4.Profitability Differences*
Although some products produce high revenue, margin percentages differ across products, meaning that not all high-revenue products are equally profitable.
## *5.Pharmacy Network Distribution*
Most pharmacies are located in Germany, France, and Italy, with the majority of stores operating in Urban and suburhan areas. Medium-sized pharmacies form the largest share of the network.
## *6.Store and Expansion*
The store age distribution shows a mix of new and established pharmacies, while store openings vary across years, indicating periods of expansion.
## *7.Business Growth*
Between 2024 and 2025, the company experienced moderate growth:
- Revenue increased by ~4.43%
- Margin increased by ~4.93%
- Units sold also increased
This suggests that sales growth was mainly driven by higher sales volume.
## *8.Profitability Stability*
Overall profit margin remained stable, increasing slightly from 27.97% in 2024 to 28.11% in 2025, indicating consistent profitability.
## *9.Category Performance Strategy*
Each product category plays a distinct role in business performance.Prescription drives revenue, Wellness maximizes profitability, and OTC drives volume, highlighting different strategic roles for each category.
 
---

# **Business Recommendations**
## *1. Focus on High-Performing Products*
Products that generate both high revenue and strong margins should be prioritized for marketing and inventory management, as they contribute significantly to overall business performance.
## *2.Improve Profitability of Lower-Margin Products*
Some high-revenue products show lower margin percentages, indicating potential cost or pricing inefficiencies. Reviewing supplier costs or pricing stratergies could improve profitability.
## *3.Strengthen Presence in High-Pharmcay Markets*
Countries such as Germany, France, and Italy, which have the largest number of pharmacies, represent key markets and should remain a focus for distribution and sales stratergies.
## *4.Leverage Urban and Suburhan Markets*
Since most pharmacies are located in urban and suburhan areas, targeted marketing and promotions in these regions may help maximize sales potential.
## *5.Monitor Sales Growth and Demand Trends*
Although the business shows moderate growth (~4-5%), continuous monitoring of monthly sales trends and product demand can help identity oppurtunities for further growth.

---

# **Final Takeways**
The analysis shows that the company's revenue and margin grew steadly from 2024-2025.
A small number of products contribute significantly to total sales, highlighting the importance of product-level performance.
Strengthening high-performing products and optimizing lower-margin items could further improve overall profitability.